# PRISM Climate Data Comprehensive Analysis — Part 3: Climate-Fire Integration

**Phase 3A** | California Wildfire Prediction Model
**Datasets**: PRISM climate (1895–2025) × CAL FIRE perimeters × MTBS severity × FPA FOD points
**Focus**: How climate drives California wildfire — the quantitative evidence

### Table of Contents

| Part | Title | Figures |
|------|-------|---------|
| 1 | Introduction & Context | — |
| 2 | Data Loading & Alignment | Fig 1 |
| 3 | Climate at Fire Locations | Fig 2–4 |
| 4 | Fire Year Climate Conditions | Fig 5–7 |
| 5 | Seasonal Fire-Climate Coupling | Fig 8–10 |
| 6 | Pre-Fire Climate Signals | Fig 11–13 |
| 7 | Fire Severity × Climate | Fig 14–15 |
| 8 | Spatial Climate-Fire Patterns | Fig 16 |
| 9 | ML Feature Summary | — |

---
## Part 1 — Introduction & Context

### The Climate-Fire Nexus

Climate is the **single most important driver** of wildfire activity in California. While topography sets the stage and vegetation provides the fuel, climate determines **when** and **how intensely** fires burn.

**Key relationships:**
- **VPD → fuel desiccation**: High VPD dries vegetation, making it flammable. VPD explains ~70% of annual burned area variance.
- **Temperature → evapotranspiration**: Higher temperatures accelerate fuel drying even without wind.
- **Precipitation → fuel moisture**: Winter/spring rain determines summer fuel conditions (antecedent moisture).
- **Precipitation → fuel growth**: Wet winters grow more fine fuels (grass), increasing subsequent fire risk.

**This notebook quantifies these relationships** using 3 independent fire datasets overlaid on 130 years of PRISM climate data.

---
## Part 2 — Data Loading & Alignment

In [1]:
# ── Imports ───────────────────────────────────────────────────────────────

import sys, warnings, gc, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec
import seaborn as sns
from scipy import stats
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.utils.constants import *
from src.utils.plotting import setup_plotting_style, save_figure
from src.utils.plotting import get_prism_variable_colors, get_vpd_risk_colors, get_drought_colors
from src.utils.prism_domain_mappings import (
    PRISM_VARIABLE_META, FIRE_WEATHER_VPD_CLASSES, CA_CLIMATE_REGIONS,
    classify_vpd, classify_drought, get_variable_label, get_variable_meta,
    compute_rh_from_tdmean_tmax, CLIMATE_SEASON_MAP
)
from src.utils.prism_data_loader import (
    check_prism_drive, read_prism_california, load_monthly_stats_ca,
    load_monthly_normal, load_all_monthly_normals,
    compute_monthly_anomaly, extract_climate_at_points,
    load_from_cache, save_to_cache
)

# Fire dataset loaders
from src.utils.data_loader import load_fire_perimeters
from src.utils.mtbs_data_loader import load_mtbs_fod, add_mtbs_derived_columns, reproject_to_albers as mtbs_reproject
from src.utils.fpa_data_loader import load_fpa_fod, add_fpa_derived_columns

setup_plotting_style()

FIGURE_DIR = OUTPUT_DIR / "figures" / "prism_fire"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Imports complete.")

# Local save helper
def save_fig(fig, name):
    """Save figure to FIGURE_DIR at 300 DPI."""
    path = FIGURE_DIR / name if not name.startswith('/') else Path(name)
    path = path.with_suffix('.png')
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Saved: {path.name} ({path.stat().st_size / 1024:.0f} KB)")
    plt.close(fig)


Imports complete.


In [2]:
# ── Load fire datasets ────────────────────────────────────────────────────

t0 = time.time()

# CAL FIRE perimeters (1878-2025, but we use 1993+)
calfire = load_fire_perimeters()
calfire_hq = calfire[calfire['YEAR_'] >= HIGH_QUALITY_CUTOFF].copy()
calfire_annual = calfire_hq.groupby('YEAR_').agg(
    n_fires=('YEAR_', 'count'),
    total_acres=('GIS_ACRES', 'sum'),
    max_acres=('GIS_ACRES', 'max'),
    mean_acres=('GIS_ACRES', 'mean')
).reset_index()
print(f"CAL FIRE: {len(calfire_hq):,} fires (1993-present), {calfire_hq.GIS_ACRES.sum()/1e6:.1f}M acres")

# MTBS centroids (1984-2024, ≥1000 ac in western US)
mtbs = load_mtbs_fod(state='CA')
mtbs = add_mtbs_derived_columns(mtbs)
mtbs_albers = mtbs_reproject(mtbs)
print(f"MTBS: {len(mtbs):,} CA fires, {mtbs.BurnBndAc.sum()/1e6:.1f}M acres, {mtbs.has_severity.sum()} with severity")

# FPA FOD (1992-2020, all sizes)
fpa = load_fpa_fod(state='CA')
fpa = add_fpa_derived_columns(fpa)
fpa_large = fpa[fpa['FIRE_SIZE'] >= 100].copy()
print(f"FPA FOD: {len(fpa):,} CA fires, {len(fpa_large):,} ≥100 acres")

# PRISM drive check
drive_info = check_prism_drive()
print(f"PRISM: {drive_info['n_variables']} variables on {drive_info['base_path']}")

# Load cached CA monthly stats
ca_stats_path = PRISM_CACHE_DIR / "prism_ca_monthly_stats.csv"
if ca_stats_path.exists():
    ca_stats = pd.read_csv(ca_stats_path)
    ca_stats['date'] = pd.to_datetime(ca_stats['year'].astype(str) + '-' + ca_stats['month'].astype(str).str.zfill(2) + '-01')
    print(f"CA monthly stats: {len(ca_stats):,} rows from cache")
else:
    print("WARNING: CA monthly stats not found — run prism_precompute_state_summaries.py --ca-only first")
    ca_stats = None

print(f"\nAll data loaded in {time.time()-t0:.0f}s")

CAL FIRE: 9,902 fires (1993-present), 24.6M acres


MTBS: 2,051 CA fires, 25.6M acres, 1581 with severity


FPA FOD: 251,881 CA fires, 5,428 ≥100 acres
PRISM: 7 variables on /Volumes/LaCie/PRISM
CA monthly stats: 11,001 rows from cache

All data loaded in 4s


In [3]:
# ── Figure 1: Temporal Alignment ──────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [1, 2]})

# Panel 1: Dataset coverage timeline
ax = axes[0]
datasets = [
    ('PRISM Monthly', 1895, 2025, '#2166ac'),
    ('PRISM Daily', 1981, 2026, '#4393c3'),
    ('MTBS', 1984, 2024, '#d6604d'),
    ('CAL FIRE (HQ)', 1993, 2025, '#b2182b'),
    ('FPA FOD', 1992, 2020, '#f4a582'),
]
for i, (name, start, end, color) in enumerate(datasets):
    ax.barh(i, end - start, left=start, height=0.6, color=color, alpha=0.85, edgecolor='white')
    ax.text(start + (end - start) / 2, i, f"{name} ({start}–{end})", ha='center', va='center',
            fontsize=9, fontweight='bold', color='white')
ax.set_yticks([])
ax.set_xlim(1890, 2030)
ax.set_xlabel('Year')
ax.set_title('Dataset Coverage Timeline', fontweight='bold', fontsize=13)
ax.axvline(1993, ls='--', color='gray', alpha=0.5, label='High-quality cutoff')
ax.legend(loc='upper left', fontsize=9)

# Panel 2: Annual fire activity vs climate
ax2 = axes[1]
if ca_stats is not None:
    # Summer (JJA) mean VPD by year
    summer_vpd = ca_stats[(ca_stats['variable'] == 'vpdmax') & (ca_stats['month'].isin([6, 7, 8]))].copy()
    annual_vpd = summer_vpd.groupby('year')['mean'].mean()
    vpd_years = annual_vpd.index[annual_vpd.index >= 1993]
    vpd_vals = annual_vpd.loc[vpd_years]

    ax2_vpd = ax2.twinx()
    ax2.bar(calfire_annual['YEAR_'], calfire_annual['total_acres'] / 1e6,
            color='#d73027', alpha=0.6, label='Burned Area (CAL FIRE)')
    ax2_vpd.plot(vpd_years, vpd_vals, color='#4575b4', lw=2.5, marker='o', ms=4,
                 label='Summer VPD (PRISM)', zorder=5)

    ax2.set_xlabel('Year', fontsize=11)
    ax2.set_ylabel('Burned Area (million acres)', color='#d73027', fontsize=11)
    ax2_vpd.set_ylabel('Summer Mean VPDmax (hPa)', color='#4575b4', fontsize=11)
    ax2.set_title('Annual Burned Area vs Summer VPD — The Climate-Fire Signal', fontweight='bold', fontsize=13)

    # Combined legend
    lines1, labels1 = ax2.get_legend_handles_labels()
    lines2, labels2 = ax2_vpd.get_legend_handles_labels()
    ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)
else:
    ax2.text(0.5, 0.5, 'CA monthly stats not available — run precomputation first',
             transform=ax2.transAxes, ha='center', va='center', fontsize=14, color='red')

plt.tight_layout()
save_fig(fig, "01_temporal_alignment")
plt.show()
print("Figure 1 saved.")

Saved: 01_temporal_alignment.png (425 KB)
Figure 1 saved.


---
## Part 3 — Climate at Fire Locations

We extract PRISM climate values at each MTBS fire centroid to compare the climate of burned locations against California's background climate.

In [4]:
# ── Extract monthly PRISM climate at MTBS centroids ───────────────────────
# For each fire, get the climate of the fire month from the nearest PRISM monthly raster

from src.utils.prism_data_loader import discover_files

# Reproject MTBS to NAD83 for PRISM extraction (PRISM uses EPSG:4269)
mtbs_nad83 = mtbs.copy()  # already in NAD83

# Sample fires: extract VPD, tmax, ppt for the fire month
fire_climate_records = []
n_extracted = 0

for _, fire in mtbs_nad83.iterrows():
    if pd.isna(fire.get('Ig_Year')) or pd.isna(fire.get('Ig_Month')):
        continue
    yr, mo = int(fire['Ig_Year']), int(fire['Ig_Month'])
    if yr < 1895 or yr > 2025:
        continue

    record = {'Event_ID': fire['Event_ID'], 'year': yr, 'month': mo,
              'BurnBndAc': fire['BurnBndAc'], 'Incid_Type': fire.get('Incid_Type', ''),
              'lat': fire.geometry.y, 'lon': fire.geometry.x}

    # Get severity if available
    for sev_col in ['Low_T_clean', 'Mod_T_clean', 'High_T_clean']:
        if sev_col in fire.index and not pd.isna(fire[sev_col]):
            record[sev_col] = fire[sev_col]

    fire_climate_records.append(record)

fire_climate = pd.DataFrame(fire_climate_records)
print(f"Prepared {len(fire_climate):,} fires for climate extraction")

# Extract climate for each unique year-month combination
unique_months = fire_climate[['year', 'month']].drop_duplicates().sort_values(['year', 'month'])
print(f"Unique year-month combinations: {len(unique_months)}")

# For each variable, extract values at fire points for each month
from src.utils.prism_data_loader import _zip_tif_path
import rasterio

climate_vars = ['vpdmax', 'tmax', 'ppt']
for var in climate_vars:
    fire_climate[var] = np.nan

t0 = time.time()
for _, row in unique_months.iterrows():
    yr, mo = int(row['year']), int(row['month'])
    mask = (fire_climate['year'] == yr) & (fire_climate['month'] == mo)
    if mask.sum() == 0:
        continue

    fires_this_month = fire_climate[mask]

    for var in climate_vars:
        files = discover_files(var, 'monthly', yr, mo)
        if not files:
            continue
        zip_path = files[0]
        try:
            tif_uri = _zip_tif_path(zip_path)
            with rasterio.open(tif_uri) as src:
                for idx in fires_this_month.index:
                    lon, lat = fire_climate.loc[idx, 'lon'], fire_climate.loc[idx, 'lat']
                    try:
                        row_px, col_px = src.index(lon, lat)
                        val = src.read(1, window=rasterio.windows.Window(col_px, row_px, 1, 1))[0, 0]
                        if val != PRISM_NODATA:
                            fire_climate.loc[idx, var] = float(val)
                    except (IndexError, ValueError):
                        pass
        except Exception:
            pass

    n_extracted += 1
    if n_extracted % 50 == 0:
        print(f"  Processed {n_extracted}/{len(unique_months)} months...")

print(f"Extraction complete in {time.time()-t0:.0f}s")
print(f"Coverage: VPD {fire_climate['vpdmax'].notna().sum()}/{len(fire_climate)}, "
      f"Tmax {fire_climate['tmax'].notna().sum()}/{len(fire_climate)}, "
      f"PPT {fire_climate['ppt'].notna().sum()}/{len(fire_climate)}")

Prepared 2,051 fires for climate extraction
Unique year-month combinations: 323


  Processed 50/323 months...


  Processed 100/323 months...


  Processed 150/323 months...


  Processed 200/323 months...


  Processed 250/323 months...


  Processed 300/323 months...


Extraction complete in 137s
Coverage: VPD 2051/2051, Tmax 2051/2051, PPT 2051/2051


In [5]:
# ── Figure 2: Climate Distributions at Fire Locations ─────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Get CA background climate from cached stats
if ca_stats is not None:
    bg_vpd = ca_stats[ca_stats['variable'] == 'vpdmax']
    bg_tmax = ca_stats[ca_stats['variable'] == 'tmax']
    bg_ppt = ca_stats[ca_stats['variable'] == 'ppt']

vars_data = [
    ('vpdmax', 'Summer VPDmax (hPa)', '#d73027', (0, 80)),
    ('tmax', 'Max Temperature (°C)', '#fc8d59', (0, 45)),
    ('ppt', 'Monthly Precipitation (mm)', '#4575b4', (0, 300)),
]

for i, (var, label, color, xlim) in enumerate(vars_data):
    # Top row: histograms
    ax = axes[0, i]
    fire_vals = fire_climate[var].dropna()
    ax.hist(fire_vals, bins=40, density=True, alpha=0.7, color=color, edgecolor='white', label='At MTBS Fires')

    if ca_stats is not None:
        bg = ca_stats[ca_stats['variable'] == var]['mean'].dropna()
        ax.hist(bg, bins=40, density=True, alpha=0.4, color='gray', edgecolor='white', label='CA Background')

    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_title(f'{label}\nat Fire Locations', fontweight='bold')
    ax.legend(fontsize=9)

    # Bottom row: CDFs
    ax2 = axes[1, i]
    sorted_fire = np.sort(fire_vals)
    cdf_fire = np.arange(1, len(sorted_fire) + 1) / len(sorted_fire)
    ax2.plot(sorted_fire, cdf_fire, color=color, lw=2, label='MTBS Fires')

    if ca_stats is not None:
        bg_vals = ca_stats[ca_stats['variable'] == var]['mean'].dropna().values
        sorted_bg = np.sort(bg_vals)
        cdf_bg = np.arange(1, len(sorted_bg) + 1) / len(sorted_bg)
        ax2.plot(sorted_bg, cdf_bg, color='gray', lw=2, ls='--', label='CA Background')

        # KS test
        ks_stat, ks_p = stats.ks_2samp(fire_vals, bg_vals)
        ax2.text(0.05, 0.95, f'KS = {ks_stat:.3f}\np < {max(ks_p, 1e-10):.1e}',
                 transform=ax2.transAxes, va='top', fontsize=10,
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax2.set_xlabel(label)
    ax2.set_ylabel('Cumulative Probability')
    ax2.set_title(f'CDF Comparison — {var}', fontweight='bold')
    ax2.legend(fontsize=9)

fig.suptitle('Climate Conditions at MTBS Fire Locations vs California Background',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, "02_climate_at_fire_locations")
plt.show()
print("Figure 2 saved.")

Saved: 02_climate_at_fire_locations.png (556 KB)
Figure 2 saved.


In [6]:
# ── Figure 3: Burned vs Unburned Climate — KS Tests ──────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Compare fire-month climate at fire locations vs same-month CA background
vars_ks = ['vpdmax', 'tmax', 'ppt']
titles = ['VPDmax (hPa)', 'Max Temperature (°C)', 'Precipitation (mm)']
colors = ['#d73027', '#fc8d59', '#4575b4']

for i, (var, title, color) in enumerate(zip(vars_ks, titles, colors)):
    ax = axes.flat[i]
    fire_vals = fire_climate[var].dropna()

    if ca_stats is not None and len(fire_vals) > 10:
        # Match by month: compare fire climate to same months in background
        fire_months = fire_climate[fire_climate[var].notna()]['month']
        bg_vals = ca_stats[(ca_stats['variable'] == var) &
                          (ca_stats['month'].isin(fire_months.unique()))]['mean'].dropna()

        # KS test
        ks_stat, ks_p = stats.ks_2samp(fire_vals, bg_vals)

        # CDFs
        sorted_f = np.sort(fire_vals)
        sorted_b = np.sort(bg_vals)
        ax.plot(sorted_f, np.linspace(0, 1, len(sorted_f)), color=color, lw=2.5,
                label=f'Burned (n={len(fire_vals):,})')
        ax.plot(sorted_b, np.linspace(0, 1, len(sorted_b)), color='gray', lw=2, ls='--',
                label=f'Background (n={len(bg_vals):,})')

        # Shade KS distance
        ax.set_title(f'{title}\nKS={ks_stat:.3f}, p<{max(ks_p, 1e-10):.1e}', fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        ax.set_title(title, fontweight='bold')

    ax.set_xlabel(title)
    ax.set_ylabel('CDF')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

# Panel 4: Summary statistics table
ax = axes[1, 1]
ax.axis('off')
if ca_stats is not None:
    summary_data = []
    for var, title in zip(vars_ks, titles):
        fv = fire_climate[var].dropna()
        bv = ca_stats[ca_stats['variable'] == var]['mean'].dropna()
        if len(fv) > 0 and len(bv) > 0:
            ks, p = stats.ks_2samp(fv, bv)
            summary_data.append([title.split('(')[0].strip(),
                                f'{fv.mean():.1f}', f'{bv.mean():.1f}',
                                f'{fv.mean() - bv.mean():.1f}', f'{ks:.3f}',
                                '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'])

    if summary_data:
        table = ax.table(cellText=summary_data,
                        colLabels=['Variable', 'Fire Mean', 'Background', 'Difference', 'KS Stat', 'Sig.'],
                        loc='center', cellLoc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(11)
        table.scale(1, 1.8)
        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_facecolor('#2166ac')
                cell.set_text_props(color='white', fontweight='bold')
            elif row % 2 == 0:
                cell.set_facecolor('#f0f0f0')
        ax.set_title('Summary: Burned vs Background Climate', fontweight='bold', fontsize=12, pad=20)

fig.suptitle('Statistical Tests: Fire Locations Experience Different Climate',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
save_fig(fig, "03_burned_vs_unburned_climate")
plt.show()
print("Figure 3 saved.")

Saved: 03_burned_vs_unburned_climate.png (528 KB)
Figure 3 saved.


In [7]:
# ── Figure 4: Fire Location Climate Map ───────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Reproject fire_climate to get lat/lon for plotting
fire_pts = fire_climate[fire_climate['vpdmax'].notna()].copy()

# Panel 1: Fires colored by local VPD
ax = axes[0]
scatter = ax.scatter(fire_pts['lon'], fire_pts['lat'],
                     c=fire_pts['vpdmax'], cmap='YlOrRd', s=fire_pts['BurnBndAc'] / 500 + 5,
                     alpha=0.6, edgecolor='gray', linewidth=0.3, vmin=0, vmax=60)
plt.colorbar(scatter, ax=ax, label='Fire-Month VPDmax (hPa)', shrink=0.7)
ax.set_xlim(-125, -114)
ax.set_ylim(32, 42.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('MTBS Fires Colored by\nFire-Month VPDmax', fontweight='bold', fontsize=13)
ax.set_aspect('equal')

# Panel 2: Fires colored by precipitation
fire_pts_ppt = fire_climate[fire_climate['ppt'].notna()].copy()
ax = axes[1]
scatter2 = ax.scatter(fire_pts_ppt['lon'], fire_pts_ppt['lat'],
                      c=fire_pts_ppt['ppt'], cmap='YlGnBu', s=fire_pts_ppt['BurnBndAc'] / 500 + 5,
                      alpha=0.6, edgecolor='gray', linewidth=0.3, vmin=0, vmax=100)
plt.colorbar(scatter2, ax=ax, label='Fire-Month Precipitation (mm)', shrink=0.7)
ax.set_xlim(-125, -114)
ax.set_ylim(32, 42.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('MTBS Fires Colored by\nFire-Month Precipitation', fontweight='bold', fontsize=13)
ax.set_aspect('equal')

fig.suptitle('Spatial Climate Context of California Wildfires (MTBS 1984–2024)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "04_fire_location_climate_map")
plt.show()
print("Figure 4 saved.")

Saved: 04_fire_location_climate_map.png (1908 KB)
Figure 4 saved.


---
## Part 4 — Fire Year Climate Conditions

Which climate conditions characterize the worst fire years? Is there a consistent climate signature for extreme fire years?

In [8]:
# ── Figure 5: Climate of Top Fire Years ───────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

if ca_stats is not None:
    # Compute annual climate metrics
    annual_climate = {}
    for var in ['vpdmax', 'tmax', 'ppt']:
        var_data = ca_stats[ca_stats['variable'] == var].copy()
        if var == 'ppt':
            annual = var_data.groupby('year')['mean'].sum()  # Annual total
        elif var == 'vpdmax':
            summer = var_data[var_data['month'].isin([6, 7, 8])]
            annual = summer.groupby('year')['mean'].mean()  # Summer mean
        else:
            annual = var_data.groupby('year')['mean'].mean()  # Annual mean
        annual_climate[var] = annual

    # Merge with fire data
    fire_years = calfire_annual.set_index('YEAR_')

    # Panel 1: Top 10 fire years annotated
    ax = axes[0, 0]
    top10 = fire_years.nlargest(10, 'total_acres')
    bars = ax.barh(range(len(top10)), top10['total_acres'] / 1e6, color='#d73027', alpha=0.8)
    ax.set_yticks(range(len(top10)))
    ax.set_yticklabels(top10.index.astype(int))
    ax.set_xlabel('Burned Area (million acres)')
    ax.set_title('Top 10 Fire Years (CAL FIRE)', fontweight='bold', fontsize=12)

    # Annotate with summer VPD
    for i, (yr, row) in enumerate(top10.iterrows()):
        if yr in annual_climate['vpdmax'].index:
            vpd = annual_climate['vpdmax'].loc[yr]
            ax.text(row['total_acres'] / 1e6 + 0.05, i, f'VPD: {vpd:.1f}',
                    va='center', fontsize=9, color='#4575b4', fontweight='bold')

    # Panel 2: Scatter — burned area vs summer VPD
    ax = axes[0, 1]
    common_years = fire_years.index.intersection(annual_climate['vpdmax'].index)
    vpd_vals = annual_climate['vpdmax'].loc[common_years]
    acres_vals = fire_years.loc[common_years, 'total_acres'] / 1e6

    ax.scatter(vpd_vals, acres_vals, c='#d73027', s=60, alpha=0.7, edgecolor='black', linewidth=0.5)

    # Label extreme years
    for yr in top10.index[:5]:
        if yr in common_years:
            ax.annotate(str(int(yr)), (vpd_vals.loc[yr], acres_vals.loc[yr]),
                       fontsize=9, fontweight='bold', xytext=(5, 5), textcoords='offset points')

    # Correlation
    r, p = stats.pearsonr(vpd_vals, acres_vals)
    slope, intercept = np.polyfit(vpd_vals, acres_vals, 1)
    x_line = np.linspace(vpd_vals.min(), vpd_vals.max(), 100)
    ax.plot(x_line, slope * x_line + intercept, 'k--', alpha=0.5)
    ax.text(0.05, 0.95, f'r = {r:.2f}, p = {p:.3f}', transform=ax.transAxes, va='top',
            fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_xlabel('Summer Mean VPDmax (hPa)', fontsize=11)
    ax.set_ylabel('Burned Area (million acres)', fontsize=11)
    ax.set_title('Burned Area vs Summer VPD', fontweight='bold', fontsize=12)

    # Panel 3: Scatter — burned area vs winter precipitation
    ax = axes[1, 0]
    # Winter = prior Dec + Jan + Feb
    winter_ppt = ca_stats[(ca_stats['variable'] == 'ppt') &
                          (ca_stats['month'].isin([12, 1, 2]))].copy()
    # Assign water year: Dec goes to next year's winter
    winter_ppt['wy'] = winter_ppt.apply(
        lambda r: r['year'] + 1 if r['month'] == 12 else r['year'], axis=1)
    annual_winter_ppt = winter_ppt.groupby('wy')['mean'].sum()

    common_yrs = fire_years.index.intersection(annual_winter_ppt.index)
    ppt_vals = annual_winter_ppt.loc[common_yrs]
    acres_v2 = fire_years.loc[common_yrs, 'total_acres'] / 1e6

    ax.scatter(ppt_vals, acres_v2, c='#4575b4', s=60, alpha=0.7, edgecolor='black', linewidth=0.5)
    r2, p2 = stats.pearsonr(ppt_vals, acres_v2)
    ax.text(0.05, 0.95, f'r = {r2:.2f}, p = {p2:.3f}', transform=ax.transAxes, va='top',
            fontsize=11, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    ax.set_xlabel('Winter Precipitation (mm, DJF)', fontsize=11)
    ax.set_ylabel('Burned Area (million acres)', fontsize=11)
    ax.set_title('Burned Area vs Winter Precipitation', fontweight='bold', fontsize=12)

    # Panel 4: Time series overlay
    ax = axes[1, 1]
    ax2_r = ax.twinx()
    common_all = fire_years.index.intersection(annual_climate['vpdmax'].index)
    ax.bar(common_all, fire_years.loc[common_all, 'total_acres'] / 1e6,
           color='#d73027', alpha=0.5, label='Burned Area')
    ax2_r.plot(common_all, annual_climate['vpdmax'].loc[common_all],
              color='#4575b4', lw=2, marker='o', ms=3, label='Summer VPD')
    ax.set_xlabel('Year')
    ax.set_ylabel('Burned Area (M acres)', color='#d73027')
    ax2_r.set_ylabel('Summer VPDmax (hPa)', color='#4575b4')
    ax.set_title('Fire Activity & Climate Co-evolution', fontweight='bold', fontsize=12)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2_r.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

fig.suptitle('Climate Conditions of California\'s Worst Fire Years', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "05_fire_year_climate")
plt.show()
print("Figure 5 saved.")

Saved: 05_fire_year_climate.png (758 KB)
Figure 5 saved.


In [9]:
# ── Figure 6: Annual Fire-Climate Correlation ─────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

if ca_stats is not None:
    # Create correlation matrix: annual fire metrics vs climate metrics
    # Build annual dataset
    annual_data = calfire_annual.set_index('YEAR_').copy()

    # Add climate variables
    for var in ['vpdmax', 'tmax', 'ppt', 'tdmean']:
        vdata = ca_stats[ca_stats['variable'] == var]
        if var == 'ppt':
            annual_data[f'{var}_annual'] = vdata.groupby('year')['mean'].sum()
        else:
            annual_data[f'{var}_annual'] = vdata.groupby('year')['mean'].mean()

        # Summer values
        summer = vdata[vdata['month'].isin([6, 7, 8])]
        annual_data[f'{var}_summer'] = summer.groupby('year')['mean'].mean()

    annual_data = annual_data.dropna()

    # Panel 1: Correlation heatmap
    ax = axes[0, 0]
    fire_cols = ['total_acres', 'n_fires', 'max_acres', 'mean_acres']
    climate_cols = ['vpdmax_summer', 'tmax_summer', 'ppt_annual', 'tdmean_summer']
    corr_matrix = annual_data[fire_cols + climate_cols].corr().loc[fire_cols, climate_cols]

    display_labels = ['Total Acres', 'Fire Count', 'Max Fire', 'Mean Fire']
    climate_labels = ['Summer VPD', 'Summer Tmax', 'Annual PPT', 'Summer Tdmean']

    sns.heatmap(corr_matrix.values, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                xticklabels=climate_labels, yticklabels=display_labels,
                ax=ax, vmin=-1, vmax=1)
    ax.set_title('Fire–Climate Correlation Matrix', fontweight='bold', fontsize=12)

    # Panel 2: Burned area vs VPD with regression
    ax = axes[0, 1]
    x = annual_data['vpdmax_summer']
    y = annual_data['total_acres'] / 1e6
    ax.scatter(x, y, c=annual_data.index, cmap='viridis', s=50, edgecolor='black', linewidth=0.5)
    slope, intercept, r, p, se = stats.linregress(x, y)
    x_fit = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_fit, slope * x_fit + intercept, 'r--', lw=2)
    ax.set_xlabel('Summer VPDmax (hPa)')
    ax.set_ylabel('Total Burned Area (M acres)')
    ax.set_title(f'VPD → Burned Area (R²={r**2:.2f})', fontweight='bold', fontsize=12)

    cbar = plt.colorbar(ax.collections[0], ax=ax, label='Year')

    # Panel 3: Rolling correlation
    ax = axes[1, 0]
    window = 10
    rolling_corr = annual_data['total_acres'].rolling(window).corr(annual_data['vpdmax_summer'])
    ax.plot(rolling_corr.index, rolling_corr.values, 'b-', lw=2)
    ax.axhline(0, ls='--', color='gray')
    ax.fill_between(rolling_corr.index, rolling_corr.values, 0, alpha=0.3,
                    where=rolling_corr.values > 0, color='red')
    ax.fill_between(rolling_corr.index, rolling_corr.values, 0, alpha=0.3,
                    where=rolling_corr.values < 0, color='blue')
    ax.set_xlabel('Year')
    ax.set_ylabel(f'{window}-Year Rolling Correlation')
    ax.set_title(f'VPD-Fire Correlation Over Time ({window}-yr window)', fontweight='bold', fontsize=12)

    # Panel 4: Ranked fire years with climate bars
    ax = axes[1, 1]
    top15 = annual_data.nlargest(15, 'total_acres')
    x_pos = np.arange(len(top15))
    width = 0.35
    ax.bar(x_pos - width/2, top15['vpdmax_summer'], width, color='#d73027', label='Summer VPD (hPa)')
    ax2_r = ax.twinx()
    ax2_r.bar(x_pos + width/2, top15['ppt_annual'], width, color='#4575b4', label='Annual PPT (mm)')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(top15.index.astype(int), rotation=45, ha='right')
    ax.set_ylabel('Summer VPDmax (hPa)', color='#d73027')
    ax2_r.set_ylabel('Annual PPT (mm)', color='#4575b4')
    ax.set_title('Climate Profile of Top 15 Fire Years', fontweight='bold', fontsize=12)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2_r.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

fig.suptitle('Annual Fire-Climate Correlations (1993–2025)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "06_annual_fire_climate_correlation")
plt.show()
print("Figure 6 saved.")

Saved: 06_annual_fire_climate_correlation.png (601 KB)
Figure 6 saved.


In [10]:
# ── Figure 7: Anatomy of the Worst Fire Year (2020) ──────────────────────

fig = plt.figure(figsize=(18, 14))
gs = GridSpec(3, 2, figure=fig, hspace=0.35, wspace=0.3)

if ca_stats is not None:
    # Get monthly climate data for 2020 and normals (1991-2020 average)
    for i, (var, label, color) in enumerate([
        ('vpdmax', 'VPDmax (hPa)', '#d73027'),
        ('ppt', 'Precipitation (mm)', '#4575b4'),
        ('tmax', 'Max Temperature (°C)', '#fc8d59'),
    ]):
        ax = fig.add_subplot(gs[i, 0])

        # 2020 monthly values
        data_2020 = ca_stats[(ca_stats['variable'] == var) & (ca_stats['year'] == 2020)].sort_values('month')
        months = data_2020['month'].values
        vals_2020 = data_2020['mean'].values

        # 30-year average (1991-2020)
        normal_data = ca_stats[(ca_stats['variable'] == var) &
                              (ca_stats['year'] >= 1991) & (ca_stats['year'] <= 2020)]
        normal_monthly = normal_data.groupby('month').agg(
            mean=('mean', 'mean'),
            std=('mean', 'std'),
            p10=('mean', lambda x: np.percentile(x, 10)),
            p90=('mean', lambda x: np.percentile(x, 90))
        )

        ax.fill_between(normal_monthly.index, normal_monthly['p10'], normal_monthly['p90'],
                        alpha=0.2, color='gray', label='Normal range (p10-p90)')
        ax.plot(normal_monthly.index, normal_monthly['mean'], 'k--', lw=1.5, label='1991-2020 Normal')
        ax.plot(months, vals_2020, color=color, lw=3, marker='o', ms=8, label='2020')

        # Shade anomaly
        for m in months:
            if m in normal_monthly.index:
                val = vals_2020[months == m][0]
                norm = normal_monthly.loc[m, 'mean']
                if var in ['vpdmax', 'tmax'] and val > norm:
                    ax.fill_between([m - 0.3, m + 0.3], norm, val, alpha=0.3, color='red')
                elif var == 'ppt' and val < norm:
                    ax.fill_between([m - 0.3, m + 0.3], val, norm, alpha=0.3, color='brown')

        ax.set_xlim(0.5, 12.5)
        ax.set_xticks(range(1, 13))
        ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
        ax.set_ylabel(label)
        ax.set_title(f'{label} — 2020 vs Normal', fontweight='bold', fontsize=12)
        ax.legend(fontsize=9, loc='upper right' if var != 'ppt' else 'upper left')
        ax.grid(True, alpha=0.3)

    # Right column: fire timeline for 2020
    ax = fig.add_subplot(gs[0, 1])
    calfire_2020 = calfire_hq[calfire_hq['YEAR_'] == 2020].copy()
    if 'ALARM_DATE' in calfire_2020.columns:
        calfire_2020['alarm_dt'] = pd.to_datetime(calfire_2020['ALARM_DATE'], errors='coerce')
        calfire_2020 = calfire_2020.dropna(subset=['alarm_dt'])
        calfire_2020['month'] = calfire_2020['alarm_dt'].dt.month
        monthly_2020 = calfire_2020.groupby('month').agg(
            n_fires=('YEAR_', 'count'),
            acres=('GIS_ACRES', 'sum')
        ).reindex(range(1, 13), fill_value=0)
        ax.bar(monthly_2020.index, monthly_2020['acres'] / 1e6, color='#d73027', alpha=0.8)
        ax.set_xticks(range(1, 13))
        ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
        ax.set_ylabel('Burned Area (M acres)')
    ax.set_title('2020 Monthly Burned Area (CAL FIRE)', fontweight='bold', fontsize=12)

    # Anomaly summary
    ax = fig.add_subplot(gs[1, 1])
    anomaly_data = []
    for var, label in [('vpdmax', 'VPDmax'), ('tmax', 'Tmax'), ('ppt', 'PPT'), ('tmean', 'Tmean')]:
        data_2020 = ca_stats[(ca_stats['variable'] == var) & (ca_stats['year'] == 2020)]
        normal_avg = ca_stats[(ca_stats['variable'] == var) &
                             (ca_stats['year'] >= 1991) & (ca_stats['year'] <= 2020)].groupby('month')['mean'].mean()

        if var == 'ppt':
            val_2020 = data_2020['mean'].sum()
            norm_val = normal_avg.sum()
        else:
            summer_2020 = data_2020[data_2020['month'].isin([6, 7, 8])]['mean'].mean()
            summer_norm = normal_avg.loc[normal_avg.index.isin([6, 7, 8])].mean()
            val_2020 = summer_2020
            norm_val = summer_norm

        anom = val_2020 - norm_val
        pct = (val_2020 / norm_val - 1) * 100 if norm_val != 0 else 0
        anomaly_data.append([label, f'{val_2020:.1f}', f'{norm_val:.1f}', f'{anom:+.1f}', f'{pct:+.1f}%'])

    if anomaly_data:
        ax.axis('off')
        table = ax.table(cellText=anomaly_data,
                        colLabels=['Variable', '2020', 'Normal', 'Anomaly', '% Diff'],
                        loc='center', cellLoc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(11)
        table.scale(1, 2)
        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_facecolor('#d73027')
                cell.set_text_props(color='white', fontweight='bold')
        ax.set_title('2020 Climate Anomalies (Summer)', fontweight='bold', fontsize=12, pad=20)

    # Historical context
    ax = fig.add_subplot(gs[2, :])
    vpd_annual = ca_stats[(ca_stats['variable'] == 'vpdmax') &
                          (ca_stats['month'].isin([6, 7, 8]))].groupby('year')['mean'].mean()
    ax.plot(vpd_annual.index, vpd_annual.values, color='gray', lw=1, alpha=0.6)
    # 10-year running mean
    running = vpd_annual.rolling(10, center=True).mean()
    ax.plot(running.index, running.values, color='#d73027', lw=2.5, label='10-year running mean')
    if 2020 in vpd_annual.index:
        ax.axvline(2020, ls='--', color='red', alpha=0.8, lw=2)
        ax.scatter([2020], [vpd_annual.loc[2020]], c='red', s=150, zorder=10,
                  edgecolor='black', linewidth=2)
        ax.annotate('2020', (2020, vpd_annual.loc[2020]), fontsize=12, fontweight='bold',
                   xytext=(10, 10), textcoords='offset points', color='red')
    ax.set_xlabel('Year')
    ax.set_ylabel('Summer Mean VPDmax (hPa)')
    ax.set_title('130-Year VPD Context: Where 2020 Sits in History', fontweight='bold', fontsize=13)
    ax.legend()

fig.suptitle('Anatomy of 2020 — California\'s Worst Fire Year', fontsize=16, fontweight='bold', y=1.01)
save_fig(fig, "07_worst_year_climate_anatomy")
plt.show()
print("Figure 7 saved.")

Saved: 07_worst_year_climate_anatomy.png (1014 KB)
Figure 7 saved.


---
## Part 5 — Seasonal Fire-Climate Coupling

Fire season in California (June–September) coincides with peak VPD and minimum precipitation. How tightly are monthly fire metrics coupled to monthly climate?

In [11]:
# ── Figure 8: Monthly Fire-Climate Coupling ───────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

if ca_stats is not None:
    # Build monthly fire counts from CAL FIRE
    calfire_monthly = calfire_hq.copy()
    if 'ALARM_DATE' in calfire_monthly.columns:
        calfire_monthly['alarm_dt'] = pd.to_datetime(calfire_monthly['ALARM_DATE'], errors='coerce')
        calfire_monthly = calfire_monthly.dropna(subset=['alarm_dt'])
        calfire_monthly['month'] = calfire_monthly['alarm_dt'].dt.month
        calfire_monthly['year'] = calfire_monthly['alarm_dt'].dt.year
    else:
        calfire_monthly['month'] = 7  # fallback
        calfire_monthly['year'] = calfire_monthly['YEAR_']

    monthly_fire_stats = calfire_monthly.groupby('month').agg(
        mean_count=('YEAR_', 'count'),
        mean_acres=('GIS_ACRES', 'sum')
    )
    monthly_fire_stats['mean_count'] = monthly_fire_stats['mean_count'] / calfire_monthly['year'].nunique()
    monthly_fire_stats['mean_acres'] = monthly_fire_stats['mean_acres'] / calfire_monthly['year'].nunique()
    monthly_fire_stats = monthly_fire_stats.reindex(range(1, 13), fill_value=0)

    # Monthly climate normals from cached stats
    climate_monthly = {}
    for var in ['vpdmax', 'ppt', 'tmax']:
        climate_monthly[var] = ca_stats[ca_stats['variable'] == var].groupby('month')['mean'].mean()

    month_labels = ['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D']

    # Panel 1: Fire count vs VPD
    ax = axes[0, 0]
    ax2 = ax.twinx()
    ax.bar(range(1, 13), monthly_fire_stats['mean_count'], color='#d73027', alpha=0.6, label='Mean Fire Count')
    ax2.plot(range(1, 13), climate_monthly['vpdmax'].loc[1:12], 'b-o', lw=2.5, label='VPDmax')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_labels)
    ax.set_ylabel('Mean Fire Count / Year', color='#d73027')
    ax2.set_ylabel('Mean VPDmax (hPa)', color='blue')
    ax.set_title('Monthly Fire Count vs VPD', fontweight='bold', fontsize=12)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    # Panel 2: Burned area vs VPD
    ax = axes[0, 1]
    ax2 = ax.twinx()
    ax.bar(range(1, 13), monthly_fire_stats['mean_acres'] / 1e3, color='#d73027', alpha=0.6, label='Mean Burned Acres')
    ax2.plot(range(1, 13), climate_monthly['vpdmax'].loc[1:12], 'b-o', lw=2.5, label='VPDmax')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_labels)
    ax.set_ylabel('Mean Burned Area (K acres)', color='#d73027')
    ax2.set_ylabel('Mean VPDmax (hPa)', color='blue')
    ax.set_title('Monthly Burned Area vs VPD', fontweight='bold', fontsize=12)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    # Panel 3: Fire count vs Precipitation (inverse)
    ax = axes[1, 0]
    ax2 = ax.twinx()
    ax.bar(range(1, 13), monthly_fire_stats['mean_count'], color='#d73027', alpha=0.6, label='Mean Fire Count')
    ax2.plot(range(1, 13), climate_monthly['ppt'].loc[1:12], 'g-o', lw=2.5, label='Precipitation')
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_labels)
    ax.set_ylabel('Mean Fire Count / Year', color='#d73027')
    ax2.set_ylabel('Mean Precipitation (mm)', color='green')
    ax.set_title('Monthly Fire Count vs Precipitation (Inverse)', fontweight='bold', fontsize=12)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

    # Panel 4: Monthly correlation bars
    ax = axes[1, 1]
    # Compute monthly correlation: monthly fire count vs monthly VPD/PPT
    monthly_corrs = {}
    for var, label in [('vpdmax', 'VPD'), ('ppt', 'PPT'), ('tmax', 'Tmax')]:
        var_monthly = ca_stats[(ca_stats['variable'] == var) &
                              (ca_stats['year'] >= 1993)].copy()
        var_monthly['ym'] = var_monthly['year'] * 100 + var_monthly['month']

        fire_monthly_ym = calfire_monthly.groupby(['year', 'month']).agg(
            count=('YEAR_', 'count'), acres=('GIS_ACRES', 'sum')).reset_index()
        fire_monthly_ym['ym'] = fire_monthly_ym['year'] * 100 + fire_monthly_ym['month']

        merged = var_monthly.merge(fire_monthly_ym[['ym', 'acres']], on='ym')
        if len(merged) > 10:
            r, p = stats.pearsonr(merged['mean'], merged['acres'])
            monthly_corrs[label] = r

    if monthly_corrs:
        bars = ax.bar(monthly_corrs.keys(), monthly_corrs.values(),
                     color=['#d73027' if v > 0 else '#4575b4' for v in monthly_corrs.values()],
                     alpha=0.8)
        ax.axhline(0, color='black', lw=0.5)
        ax.set_ylabel('Pearson Correlation')
        ax.set_title('Monthly Climate–Fire Correlation', fontweight='bold', fontsize=12)
        for bar, val in zip(bars, monthly_corrs.values()):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                   f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

fig.suptitle('Seasonal Fire-Climate Coupling in California', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "08_monthly_fire_climate_coupling")
plt.show()
print("Figure 8 saved.")

Saved: 08_monthly_fire_climate_coupling.png (637 KB)
Figure 8 saved.


In [12]:
# ── Figure 9: Fire Season Climate Trends ──────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

if ca_stats is not None:
    fire_months = [6, 7, 8, 9]  # June-September

    # Panel 1: Fire season VPD trend
    ax = axes[0, 0]
    vpd_fire_season = ca_stats[(ca_stats['variable'] == 'vpdmax') &
                               (ca_stats['month'].isin(fire_months))]
    annual_vpd_fs = vpd_fire_season.groupby('year')['mean'].mean()
    years = annual_vpd_fs.index[annual_vpd_fs.index >= 1950]
    vals = annual_vpd_fs.loc[years]

    ax.plot(years, vals, color='gray', alpha=0.5, lw=1)
    running = vals.rolling(10, center=True).mean()
    ax.plot(running.index, running.values, color='#d73027', lw=2.5, label='10-yr mean')

    # Trend line
    slope, intercept, r, p, se = stats.linregress(years.astype(float), vals.values)
    ax.plot(years, slope * years.astype(float) + intercept, 'k--', lw=1.5,
            label=f'Trend: {slope*10:.2f} hPa/decade')
    ax.set_xlabel('Year')
    ax.set_ylabel('Fire Season VPDmax (hPa)')
    ax.set_title(f'Fire Season (JJAS) VPD Trend\n+{slope*10:.2f} hPa/decade, p={p:.3f}',
                fontweight='bold', fontsize=12)
    ax.legend()

    # Panel 2: Fire season PPT trend
    ax = axes[0, 1]
    ppt_fire_season = ca_stats[(ca_stats['variable'] == 'ppt') &
                               (ca_stats['month'].isin(fire_months))]
    annual_ppt_fs = ppt_fire_season.groupby('year')['mean'].sum()
    years_ppt = annual_ppt_fs.index[annual_ppt_fs.index >= 1950]
    vals_ppt = annual_ppt_fs.loc[years_ppt]

    ax.plot(years_ppt, vals_ppt, color='gray', alpha=0.5, lw=1)
    running_ppt = vals_ppt.rolling(10, center=True).mean()
    ax.plot(running_ppt.index, running_ppt.values, color='#4575b4', lw=2.5, label='10-yr mean')
    slope2, _, r2, p2, _ = stats.linregress(years_ppt.astype(float), vals_ppt.values)
    ax.plot(years_ppt, slope2 * years_ppt.astype(float) + intercept, 'k--', lw=1.5,
            label=f'Trend: {slope2*10:.2f} mm/decade')
    ax.set_xlabel('Year')
    ax.set_ylabel('Fire Season PPT (mm)')
    ax.set_title(f'Fire Season (JJAS) Precipitation Trend', fontweight='bold', fontsize=12)
    ax.legend()

    # Panel 3: Fire season VPD overlaid with fire activity
    ax = axes[1, 0]
    ax2 = ax.twinx()
    common = annual_vpd_fs.index.intersection(calfire_annual.set_index('YEAR_').index)
    ax.bar(common, calfire_annual.set_index('YEAR_').loc[common, 'total_acres'] / 1e6,
           color='#d73027', alpha=0.5, label='Burned Area')
    ax2.plot(annual_vpd_fs.loc[common].index, annual_vpd_fs.loc[common].values,
            color='#4575b4', lw=2, label='Fire Season VPD')
    ax.set_xlabel('Year')
    ax.set_ylabel('Burned Area (M acres)', color='#d73027')
    ax2.set_ylabel('Fire Season VPDmax (hPa)', color='#4575b4')
    ax.set_title('Fire Activity & Fire Season VPD', fontweight='bold', fontsize=12)
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    # Panel 4: Fire season Tmax trend
    ax = axes[1, 1]
    tmax_fs = ca_stats[(ca_stats['variable'] == 'tmax') &
                       (ca_stats['month'].isin(fire_months))]
    annual_tmax_fs = tmax_fs.groupby('year')['mean'].mean()
    years_t = annual_tmax_fs.index[annual_tmax_fs.index >= 1950]
    vals_t = annual_tmax_fs.loc[years_t]

    ax.plot(years_t, vals_t, color='gray', alpha=0.5, lw=1)
    running_t = vals_t.rolling(10, center=True).mean()
    ax.plot(running_t.index, running_t.values, color='#fc8d59', lw=2.5, label='10-yr mean')
    slope3, _, r3, p3, _ = stats.linregress(years_t.astype(float), vals_t.values)
    ax.set_xlabel('Year')
    ax.set_ylabel('Fire Season Tmax (°C)')
    ax.set_title(f'Fire Season (JJAS) Temperature Trend\n+{slope3*10:.2f} °C/decade, p={p3:.3f}',
                fontweight='bold', fontsize=12)
    ax.legend()

fig.suptitle('Fire Season Climate Trends (1950–2025)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "09_fire_season_climate_trends")
plt.show()
print("Figure 9 saved.")

Saved: 09_fire_season_climate_trends.png (1146 KB)
Figure 9 saved.


In [13]:
# ── Figure 10: Fire Season VPD vs Burned Acres Scatter ────────────────────

fig, ax = plt.subplots(figsize=(14, 10))

if ca_stats is not None:
    # Fire season (JJAS) mean VPD
    vpd_fs = ca_stats[(ca_stats['variable'] == 'vpdmax') &
                      (ca_stats['month'].isin([6, 7, 8, 9]))].groupby('year')['mean'].mean()

    # Merge with fire data
    fire_yrs = calfire_annual.set_index('YEAR_')
    common = vpd_fs.index.intersection(fire_yrs.index)
    x = vpd_fs.loc[common]
    y = fire_yrs.loc[common, 'total_acres'] / 1e6

    # Size by max fire
    sizes = fire_yrs.loc[common, 'max_acres'] / 5000 + 30

    scatter = ax.scatter(x, y, c=common, cmap='plasma', s=sizes,
                        alpha=0.75, edgecolor='black', linewidth=0.8)

    # Label notable years
    notable = [2020, 2021, 2018, 2017, 2008, 2007, 2003, 2002]
    for yr in notable:
        if yr in common:
            ax.annotate(str(yr), (x.loc[yr], y.loc[yr]),
                       fontsize=10, fontweight='bold',
                       xytext=(8, 8), textcoords='offset points',
                       arrowprops=dict(arrowstyle='->', color='gray'))

    # Regression
    slope, intercept, r, p, se = stats.linregress(x.astype(float), y.values)
    x_fit = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_fit, slope * x_fit + intercept, 'r--', lw=2,
            label=f'Linear fit (R²={r**2:.2f}, p={p:.3f})')

    # VPD threshold annotation
    ax.axvline(PRISM_VPD_CRITICAL_HPA, ls=':', color='red', alpha=0.7, lw=2)
    ax.text(PRISM_VPD_CRITICAL_HPA + 0.5, ax.get_ylim()[1] * 0.95,
            f'Critical VPD\n({PRISM_VPD_CRITICAL_HPA} hPa)',
            color='red', fontsize=10, va='top')

    plt.colorbar(scatter, ax=ax, label='Year', shrink=0.8)
    ax.set_xlabel('Fire Season Mean VPDmax (hPa)', fontsize=13)
    ax.set_ylabel('Annual Burned Area (million acres)', fontsize=13)
    ax.set_title('The VPD–Fire Relationship: Fire Season VPD Predicts Burned Area',
                fontweight='bold', fontsize=14)
    ax.legend(fontsize=11, loc='upper left')
    ax.grid(True, alpha=0.3)

save_fig(fig, "10_fire_season_vpd_vs_acres")
plt.show()
print("Figure 10 saved.")

Saved: 10_fire_season_vpd_vs_acres.png (295 KB)
Figure 10 saved.


---
## Part 6 — Pre-Fire Climate Signals

Can we predict fire activity from antecedent (prior months') climate? The "spring drought → summer fire" hypothesis is tested here with lag correlations.

In [14]:
# ── Figure 11: Antecedent Climate Signals ─────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

if ca_stats is not None:
    # Build annual burned area
    fire_annual = calfire_annual.set_index('YEAR_')['total_acres']

    # For each lag (1-6 months before peak fire season Aug), compute correlation
    fire_peak_month = 8  # August

    for col_idx, (var, label, color) in enumerate([
        ('vpdmax', 'VPDmax', '#d73027'),
        ('ppt', 'Precipitation', '#4575b4'),
        ('tmax', 'Temperature', '#fc8d59'),
    ]):
        # Top row: lag correlations
        ax = axes[0, col_idx]
        lag_corrs = []
        lag_pvals = []
        for lag in range(0, 7):
            target_month = fire_peak_month - lag
            target_year_offset = 0
            if target_month <= 0:
                target_month += 12
                target_year_offset = -1

            var_data = ca_stats[(ca_stats['variable'] == var) &
                               (ca_stats['month'] == target_month)].copy()
            var_data['fire_year'] = var_data['year'] + target_year_offset

            merged = var_data.set_index('fire_year')['mean'].to_frame().join(fire_annual)
            merged = merged.dropna()

            if len(merged) > 10:
                r, p = stats.pearsonr(merged['mean'], merged['total_acres'])
                lag_corrs.append(r)
                lag_pvals.append(p)
            else:
                lag_corrs.append(0)
                lag_pvals.append(1)

        bars = ax.bar(range(0, 7), lag_corrs,
                     color=[color if p < 0.05 else 'lightgray' for p in lag_pvals],
                     alpha=0.8, edgecolor='black', linewidth=0.5)
        ax.axhline(0, color='black', lw=0.5)
        ax.set_xticks(range(0, 7))
        ax.set_xticklabels(['Aug\n(t=0)', 'Jul\n(t-1)', 'Jun\n(t-2)', 'May\n(t-3)', 'Apr\n(t-4)', 'Mar\n(t-5)', 'Feb\n(t-6)'])
        ax.set_ylabel('Correlation with Annual Burned Area')
        ax.set_title(f'{label} Lag Correlation', fontweight='bold', fontsize=12)
        ax.set_ylim(-0.5, 0.7)

        # Significance markers
        for j, (c, p) in enumerate(zip(lag_corrs, lag_pvals)):
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            if sig:
                ax.text(j, c + 0.03 * np.sign(c), sig, ha='center', fontsize=12, fontweight='bold')

        # Bottom row: time series comparison
        ax2 = axes[1, col_idx]
        # Show the most correlated lag
        best_lag = np.argmax(np.abs(lag_corrs))
        best_month = fire_peak_month - best_lag
        if best_month <= 0:
            best_month += 12

        var_best = ca_stats[(ca_stats['variable'] == var) &
                           (ca_stats['month'] == best_month)].copy()
        if best_lag > 0 and best_month > fire_peak_month:
            var_best['fire_year'] = var_best['year'] - 1
        else:
            var_best['fire_year'] = var_best['year']

        common = var_best.set_index('fire_year')['mean'].to_frame().join(fire_annual).dropna()

        if len(common) > 10:
            ax2_r = ax2.twinx()
            ax2.bar(common.index, common['total_acres'] / 1e6, color='#d73027', alpha=0.4, label='Burned Area')
            ax2_r.plot(common.index, common['mean'], color=color, lw=2, label=f'{label} (month {best_month})')
            ax2.set_xlabel('Year')
            ax2.set_ylabel('Burned Area (M acres)', color='#d73027')
            ax2_r.set_ylabel(f'{label}', color=color)
            ax2.set_title(f'Best Lag: Month {best_month} (lag={best_lag})', fontweight='bold')
            lines1, labels1 = ax2.get_legend_handles_labels()
            lines2, labels2 = ax2_r.get_legend_handles_labels()
            ax2.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=9)

fig.suptitle('Antecedent Climate Signals: Can Prior Months Predict Fire?',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "11_antecedent_climate_signals")
plt.show()
print("Figure 11 saved.")

Saved: 11_antecedent_climate_signals.png (782 KB)
Figure 11 saved.


In [15]:
# ── Figure 12: Spring Drought → Summer Fire ───────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

if ca_stats is not None:
    fire_annual = calfire_annual.set_index('YEAR_')

    # Compute spring (MAM) precipitation deficit
    spring_ppt = ca_stats[(ca_stats['variable'] == 'ppt') &
                          (ca_stats['month'].isin([3, 4, 5]))].groupby('year')['mean'].sum()
    spring_normal = spring_ppt.mean()
    spring_deficit = spring_ppt - spring_normal

    # Panel 1: Spring PPT deficit vs summer burned area
    ax = axes[0, 0]
    common = spring_deficit.index.intersection(fire_annual.index)
    x = spring_deficit.loc[common]
    y = fire_annual.loc[common, 'total_acres'] / 1e6

    colors_scatter = ['#d73027' if xi < 0 else '#4575b4' for xi in x]
    ax.scatter(x, y, c=colors_scatter, s=60, alpha=0.7, edgecolor='black', linewidth=0.5)

    r, p = stats.pearsonr(x, y)
    ax.text(0.05, 0.95, f'r = {r:.2f}, p = {p:.3f}', transform=ax.transAxes, va='top',
            fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

    ax.axvline(0, ls='--', color='gray')
    ax.set_xlabel('Spring (MAM) Precipitation Anomaly (mm)')
    ax.set_ylabel('Annual Burned Area (M acres)')
    ax.set_title('Spring Drought → Summer Fire', fontweight='bold', fontsize=12)

    # Label extreme cases
    for yr in [2020, 2021, 2018, 2017, 2012]:
        if yr in common:
            ax.annotate(str(yr), (x.loc[yr], y.loc[yr]), fontsize=9, fontweight='bold',
                       xytext=(5, 5), textcoords='offset points')

    # Panel 2: Dry winter + hot summer → fire
    ax = axes[0, 1]
    winter_ppt = ca_stats[(ca_stats['variable'] == 'ppt') &
                          (ca_stats['month'].isin([12, 1, 2]))].copy()
    winter_ppt['wy'] = winter_ppt.apply(lambda r: r['year'] + 1 if r['month'] == 12 else r['year'], axis=1)
    annual_winter = winter_ppt.groupby('wy')['mean'].sum()

    summer_vpd = ca_stats[(ca_stats['variable'] == 'vpdmax') &
                          (ca_stats['month'].isin([6, 7, 8]))].groupby('year')['mean'].mean()

    common3 = annual_winter.index.intersection(summer_vpd.index).intersection(fire_annual.index)
    if len(common3) > 10:
        scatter = ax.scatter(annual_winter.loc[common3], summer_vpd.loc[common3],
                           c=fire_annual.loc[common3, 'total_acres'] / 1e6,
                           cmap='YlOrRd', s=80, edgecolor='black', linewidth=0.5)
        plt.colorbar(scatter, ax=ax, label='Burned Area (M acres)', shrink=0.8)
        ax.set_xlabel('Winter PPT (mm, DJF)')
        ax.set_ylabel('Summer VPDmax (hPa)')
        ax.set_title('Winter Precip × Summer VPD\n(colored by burned area)', fontweight='bold', fontsize=12)

        # Quadrant labels
        med_x, med_y = annual_winter.loc[common3].median(), summer_vpd.loc[common3].median()
        ax.axvline(med_x, ls=':', color='gray', alpha=0.5)
        ax.axhline(med_y, ls=':', color='gray', alpha=0.5)
        ax.text(0.02, 0.98, 'Dry+Hot\n(worst)', transform=ax.transAxes, va='top', fontsize=10, color='red')
        ax.text(0.98, 0.02, 'Wet+Cool\n(best)', transform=ax.transAxes, ha='right', fontsize=10, color='blue')

    # Panel 3: Case studies — annotated timeline
    ax = axes[1, 0]
    drought_years = [2002, 2007, 2008, 2012, 2017, 2018, 2020, 2021]
    annual_burned = fire_annual['total_acres'] / 1e6

    ax.bar(annual_burned.index, annual_burned.values, color='#d73027', alpha=0.5)
    for yr in drought_years:
        if yr in annual_burned.index:
            ax.annotate(str(yr), (yr, annual_burned.loc[yr]), fontsize=8, fontweight='bold',
                       rotation=45, ha='left', va='bottom', color='darkred')
            ax.bar(yr, annual_burned.loc[yr], color='darkred', alpha=0.9)

    ax.set_xlabel('Year')
    ax.set_ylabel('Burned Area (M acres)')
    ax.set_title('Major Fire Years Highlighted', fontweight='bold', fontsize=12)

    # Panel 4: 2-year lookback — cumulative VPD anomaly
    ax = axes[1, 1]
    vpd_annual = ca_stats[(ca_stats['variable'] == 'vpdmax')].groupby('year')['mean'].mean()
    vpd_normal = vpd_annual.mean()
    vpd_anomaly = vpd_annual - vpd_normal
    vpd_2yr = vpd_anomaly.rolling(2).sum()

    common4 = vpd_2yr.dropna().index.intersection(fire_annual.index)
    ax.scatter(vpd_2yr.loc[common4], fire_annual.loc[common4, 'total_acres'] / 1e6,
              c='#d73027', s=50, alpha=0.7, edgecolor='black', linewidth=0.5)
    r4, p4 = stats.pearsonr(vpd_2yr.loc[common4], fire_annual.loc[common4, 'total_acres'])
    ax.text(0.05, 0.95, f'r = {r4:.2f}, p = {p4:.3f}', transform=ax.transAxes, va='top',
            fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_xlabel('2-Year Cumulative VPD Anomaly (hPa)')
    ax.set_ylabel('Burned Area (M acres)')
    ax.set_title('Cumulative VPD Stress → Fire', fontweight='bold', fontsize=12)

fig.suptitle('Pre-Fire Climate Signals: Antecedent Drought Drives Fire',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "12_spring_drought_summer_fire")
plt.show()
print("Figure 12 saved.")

Saved: 12_spring_drought_summer_fire.png (613 KB)
Figure 12 saved.


In [16]:
# ── Figure 13: Cumulative VPD and Fire Risk ───────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

if ca_stats is not None:
    # Build monthly VPD with fire data
    vpd_monthly = ca_stats[ca_stats['variable'] == 'vpdmax'].copy()
    vpd_monthly['ym'] = vpd_monthly['year'] * 100 + vpd_monthly['month']

    # Fire activity by month
    if 'ALARM_DATE' in calfire_hq.columns:
        calfire_monthly = calfire_hq.copy()
        calfire_monthly['alarm_dt'] = pd.to_datetime(calfire_monthly['ALARM_DATE'], errors='coerce')
        calfire_monthly = calfire_monthly.dropna(subset=['alarm_dt'])
        calfire_monthly['year'] = calfire_monthly['alarm_dt'].dt.year
        calfire_monthly['month'] = calfire_monthly['alarm_dt'].dt.month
        calfire_monthly['ym'] = calfire_monthly['year'] * 100 + calfire_monthly['month']

        fire_by_ym = calfire_monthly.groupby('ym').agg(
            fire_count=('YEAR_', 'count'),
            acres=('GIS_ACRES', 'sum')
        ).reset_index()

        # Merge
        merged = vpd_monthly.merge(fire_by_ym, on='ym', how='left')
        merged['fire_count'] = merged['fire_count'].fillna(0)
        merged['acres'] = merged['acres'].fillna(0)
        merged['has_fire'] = merged['fire_count'] > 0

        # Panel 1: VPD bins vs fire probability
        ax = axes[0]
        vpd_bins = np.arange(0, 65, 5)
        merged['vpd_bin'] = pd.cut(merged['mean'], vpd_bins)
        fire_prob = merged.groupby('vpd_bin').agg(
            n_months=('has_fire', 'count'),
            n_fire_months=('has_fire', 'sum'),
            mean_acres=('acres', 'mean')
        )
        fire_prob['probability'] = fire_prob['n_fire_months'] / fire_prob['n_months']
        fire_prob = fire_prob[fire_prob['n_months'] >= 5]

        bin_centers = [interval.mid for interval in fire_prob.index]
        ax.bar(bin_centers, fire_prob['probability'], width=4, color='#d73027', alpha=0.7, edgecolor='white')
        ax.set_xlabel('Monthly Mean VPDmax (hPa)', fontsize=12)
        ax.set_ylabel('Probability of Fire Activity', fontsize=12)
        ax.set_title('Fire Probability Increases Sharply with VPD', fontweight='bold', fontsize=13)
        ax.axvline(PRISM_VPD_CRITICAL_HPA, ls=':', color='red', lw=2, label=f'Critical VPD ({PRISM_VPD_CRITICAL_HPA} hPa)')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)

        # Panel 2: Cumulative VPD vs mean fire size
        ax = axes[1]
        ax2 = ax.twinx()
        ax.bar(bin_centers, fire_prob['mean_acres'] / 1e3, width=4, color='#fc8d59', alpha=0.7,
               edgecolor='white', label='Mean Fire Size')
        ax2.plot(bin_centers, fire_prob['n_fire_months'], 'b-o', lw=2, label='Fire Month Count')
        ax.set_xlabel('Monthly Mean VPDmax (hPa)', fontsize=12)
        ax.set_ylabel('Mean Monthly Burned Area (K acres)', color='#fc8d59', fontsize=12)
        ax2.set_ylabel('Number of Fire Months', color='blue', fontsize=12)
        ax.set_title('Fire Size and Frequency Both Scale with VPD', fontweight='bold', fontsize=13)
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

fig.suptitle('VPD as a Fire Risk Predictor', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "13_cumulative_vpd_fire_risk")
plt.show()
print("Figure 13 saved.")

Saved: 13_cumulative_vpd_fire_risk.png (405 KB)
Figure 13 saved.


---
## Part 7 — Fire Severity × Climate

Do hotter, drier conditions produce more severe fires? Using MTBS burn severity thresholds (dNBR), we test whether climate explains fire severity.

In [17]:
# ── Figure 14: Severity vs Climate ────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Merge fire climate data with severity
fire_sev = fire_climate[fire_climate['High_T_clean'].notna()].copy()
print(f"Fires with both climate and severity data: {len(fire_sev)}")

if len(fire_sev) > 20:
    # Panel 1: Scatter VPD vs High_T severity
    ax = axes[0, 0]
    valid = fire_sev[fire_sev['vpdmax'].notna()]
    ax.scatter(valid['vpdmax'], valid['High_T_clean'], c='#d73027', s=valid['BurnBndAc']/500+10,
              alpha=0.5, edgecolor='gray', linewidth=0.3)
    if len(valid) > 10:
        r, p = stats.pearsonr(valid['vpdmax'], valid['High_T_clean'])
        slope, intercept = np.polyfit(valid['vpdmax'], valid['High_T_clean'], 1)
        x_fit = np.linspace(valid['vpdmax'].min(), valid['vpdmax'].max(), 100)
        ax.plot(x_fit, slope * x_fit + intercept, 'k--', lw=2)
        ax.text(0.05, 0.95, f'r = {r:.2f}, p = {p:.3f}', transform=ax.transAxes, va='top',
                fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    ax.set_xlabel('Fire-Month VPDmax (hPa)', fontsize=11)
    ax.set_ylabel('High Severity Threshold (dNBR)', fontsize=11)
    ax.set_title('VPD vs Burn Severity', fontweight='bold', fontsize=12)

    # Panel 2: Scatter PPT vs High_T
    ax = axes[0, 1]
    valid_ppt = fire_sev[fire_sev['ppt'].notna()]
    ax.scatter(valid_ppt['ppt'], valid_ppt['High_T_clean'], c='#4575b4', s=valid_ppt['BurnBndAc']/500+10,
              alpha=0.5, edgecolor='gray', linewidth=0.3)
    if len(valid_ppt) > 10:
        r2, p2 = stats.pearsonr(valid_ppt['ppt'], valid_ppt['High_T_clean'])
        ax.text(0.05, 0.95, f'r = {r2:.2f}, p = {p2:.3f}', transform=ax.transAxes, va='top',
                fontsize=11, bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
    ax.set_xlabel('Fire-Month Precipitation (mm)', fontsize=11)
    ax.set_ylabel('High Severity Threshold (dNBR)', fontsize=11)
    ax.set_title('Precipitation vs Burn Severity', fontweight='bold', fontsize=12)

    # Panel 3: Severity boxplots by VPD class
    ax = axes[1, 0]
    valid_vpd = fire_sev[fire_sev['vpdmax'].notna()].copy()
    vpd_classes = classify_vpd(valid_vpd['vpdmax'].values)
    valid_vpd['vpd_class'] = vpd_classes

    class_order = ['Minimal', 'Low', 'Moderate', 'High', 'Extreme', 'Exceptional']
    existing_classes = [c for c in class_order if c in valid_vpd['vpd_class'].unique()]
    vpd_colors = get_vpd_risk_colors()

    box_data = [valid_vpd[valid_vpd['vpd_class'] == c]['High_T_clean'].dropna() for c in existing_classes]
    bp = ax.boxplot(box_data, labels=existing_classes, patch_artist=True, widths=0.6)
    for patch, cls in zip(bp['boxes'], existing_classes):
        patch.set_facecolor(vpd_colors.get(cls, '#cccccc'))
        patch.set_alpha(0.7)
    ax.set_xlabel('VPD Risk Class')
    ax.set_ylabel('High Severity Threshold (dNBR)')
    ax.set_title('Burn Severity by VPD Risk Class', fontweight='bold', fontsize=12)
    ax.tick_params(axis='x', rotation=30)

    # Panel 4: Summary table
    ax = axes[1, 1]
    ax.axis('off')
    summary = []
    for cls in existing_classes:
        subset = valid_vpd[valid_vpd['vpd_class'] == cls]['High_T_clean']
        if len(subset) > 0:
            summary.append([cls, f'{len(subset)}', f'{subset.mean():.0f}',
                          f'{subset.median():.0f}', f'{subset.std():.0f}'])
    if summary:
        table = ax.table(cellText=summary,
                        colLabels=['VPD Class', 'N Fires', 'Mean dNBR', 'Median', 'Std Dev'],
                        loc='center', cellLoc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(11)
        table.scale(1, 2)
        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_facecolor('#4a1486')
                cell.set_text_props(color='white', fontweight='bold')
        ax.set_title('Severity Statistics by VPD Class', fontweight='bold', fontsize=12, pad=20)

fig.suptitle('Fire Severity × Climate Conditions (MTBS 1984–2024)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "14_severity_vs_climate")
plt.show()
print("Figure 14 saved.")

Fires with both climate and severity data: 1581


Saved: 14_severity_vs_climate.png (1428 KB)
Figure 14 saved.


In [18]:
# ── Figure 15: Severity Climate Trends ────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Panel 1: Mean severity by year with climate overlay
fire_sev_annual = fire_sev.groupby('year').agg(
    mean_high_t=('High_T_clean', 'mean'),
    n_fires=('High_T_clean', 'count')
).reset_index()
fire_sev_annual = fire_sev_annual[fire_sev_annual['n_fires'] >= 3]

ax = axes[0]
ax2 = ax.twinx()

ax.bar(fire_sev_annual['year'], fire_sev_annual['mean_high_t'],
       color='#e41a1c', alpha=0.6, label='Mean High_T (dNBR)')

if ca_stats is not None:
    vpd_annual = ca_stats[(ca_stats['variable'] == 'vpdmax') &
                          (ca_stats['month'].isin([6, 7, 8]))].groupby('year')['mean'].mean()
    common = fire_sev_annual.set_index('year').index.intersection(vpd_annual.index)
    ax2.plot(common, vpd_annual.loc[common], 'b-o', ms=4, lw=2, label='Summer VPD')

ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Mean High Severity Threshold (dNBR)', color='#e41a1c', fontsize=11)
ax2.set_ylabel('Summer Mean VPDmax (hPa)', color='blue', fontsize=11)
ax.set_title('Fire Severity Trends with Climate Warming Signal', fontweight='bold', fontsize=13)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Panel 2: Decadal severity by VPD regime
ax = axes[1]
fire_sev_decade = fire_sev.copy()
fire_sev_decade['decade'] = (fire_sev_decade['year'] // 10) * 10

decade_stats = fire_sev_decade.groupby('decade').agg(
    mean_severity=('High_T_clean', 'mean'),
    n_fires=('High_T_clean', 'count'),
    mean_vpd=('vpdmax', 'mean')
).dropna()

x = np.arange(len(decade_stats))
width = 0.35
ax.bar(x - width/2, decade_stats['mean_severity'], width, color='#e41a1c', alpha=0.7, label='Mean Severity (dNBR)')
ax_r = ax.twinx()
ax_r.bar(x + width/2, decade_stats['mean_vpd'], width, color='#4575b4', alpha=0.7, label='Mean VPD (hPa)')

ax.set_xticks(x)
ax.set_xticklabels([f"{int(d)}s\n(n={int(n)})" for d, n in zip(decade_stats.index, decade_stats['n_fires'])])
ax.set_ylabel('Mean dNBR High_T', color='#e41a1c')
ax_r.set_ylabel('Mean Fire-Month VPD (hPa)', color='#4575b4')
ax.set_title('Decadal Evolution: Severity and VPD Co-Increase', fontweight='bold', fontsize=13)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax_r.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
save_fig(fig, "15_severity_climate_trends")
plt.show()
print("Figure 15 saved.")

Saved: 15_severity_climate_trends.png (473 KB)
Figure 15 saved.


---
## Part 8 — Spatial Climate-Fire Patterns

Where in California do the worst fire-climate conditions coincide? A composite map of fire density weighted by local VPD.

In [19]:
# ── Figure 16: Climate-Fire Spatial Patterns ──────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 10))

# Panel 1: Fire density colored by mean summer VPD
ax = axes[0]
# Bin MTBS fires into a grid and color by mean VPD
fire_pts_vpd = fire_climate[fire_climate['vpdmax'].notna()].copy()

# Create 2D histogram
lon_bins = np.linspace(-125, -114, 50)
lat_bins = np.linspace(32, 42.5, 50)

# Fire count per bin
h_count, xedges, yedges = np.histogram2d(fire_pts_vpd['lon'], fire_pts_vpd['lat'],
                                          bins=[lon_bins, lat_bins])

# Mean VPD per bin
h_vpd = np.zeros_like(h_count)
h_n = np.zeros_like(h_count)
for _, fire in fire_pts_vpd.iterrows():
    xi = np.searchsorted(lon_bins, fire['lon']) - 1
    yi = np.searchsorted(lat_bins, fire['lat']) - 1
    if 0 <= xi < len(lon_bins)-1 and 0 <= yi < len(lat_bins)-1:
        h_vpd[xi, yi] += fire['vpdmax']
        h_n[xi, yi] += 1

mean_vpd_grid = np.where(h_n > 0, h_vpd / h_n, np.nan)

# Plot
im = ax.pcolormesh(xedges, yedges, mean_vpd_grid.T, cmap='YlOrRd', vmin=10, vmax=50)
plt.colorbar(im, ax=ax, label='Mean Fire-Month VPDmax (hPa)', shrink=0.7)

# Overlay fire count contours
h_count_masked = np.where(h_count > 0, h_count, np.nan)
ax.contour(0.5*(lon_bins[:-1]+lon_bins[1:]), 0.5*(lat_bins[:-1]+lat_bins[1:]),
           h_count_masked.T, levels=[2, 5, 10, 20], colors='black', linewidths=1, alpha=0.5)

ax.set_xlim(-125, -114)
ax.set_ylim(32, 42.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Fire Hotspot VPD\n(contours = fire count)', fontweight='bold', fontsize=13)
ax.set_aspect('equal')

# Panel 2: Fire size colored by VPD
ax = axes[1]
# Sort so largest fires plot last (on top)
fire_sorted = fire_pts_vpd.sort_values('BurnBndAc')
scatter = ax.scatter(fire_sorted['lon'], fire_sorted['lat'],
                    c=fire_sorted['vpdmax'], cmap='YlOrRd',
                    s=np.log10(fire_sorted['BurnBndAc'] + 1) * 15,
                    alpha=0.6, edgecolor='gray', linewidth=0.3,
                    vmin=10, vmax=50)
plt.colorbar(scatter, ax=ax, label='Fire-Month VPDmax (hPa)', shrink=0.7)

# Annotate top 5 largest fires
top5 = fire_sorted.nlargest(5, 'BurnBndAc')
for _, fire in top5.iterrows():
    name = fire.get('Event_ID', '')[:8]
    ax.annotate(f"{name}\n{int(fire['BurnBndAc']):,}ac",
               (fire['lon'], fire['lat']),
               fontsize=7, fontweight='bold',
               xytext=(10, 10), textcoords='offset points',
               arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlim(-125, -114)
ax.set_ylim(32, 42.5)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('MTBS Fires: Size & VPD\n(point size = log(acres))', fontweight='bold', fontsize=13)
ax.set_aspect('equal')

fig.suptitle('Spatial Climate-Fire Patterns: Where Heat Meets Fire in California',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig(fig, "16_climate_fire_spatial_patterns")
plt.show()
print("Figure 16 saved.")

Saved: 16_climate_fire_spatial_patterns.png (1665 KB)
Figure 16 saved.


---
## Part 9 — ML Feature Summary & Executive Summary

### Climate-Derived ML Features

This analysis demonstrates that PRISM climate data provides **the strongest predictive signal** for California wildfire. The table below summarizes the 30+ features available for the ML model:

| Feature Group | Variables | Temporal Design | Importance |
|---------------|-----------|-----------------|------------|
| **Monthly VPD** | VPDmax, VPDmin | Current month + 1-6 month lags | Very High |
| **Temperature** | Tmax, Tmin, Tmean, DTR | Current + lags | High |
| **Precipitation** | PPT | Current + cumulative (3/6/12 month) | High |
| **Moisture** | Tdmean, est. RH, DPD | Current month | Moderate-High |
| **Anomalies** | All vars vs 30-yr normals | Current + lags | High |
| **Seasonal** | Fire season means, winter PPT total | Seasonal aggregation | Very High |
| **Cumulative stress** | Rolling VPD sum, dry days, PPT deficit | 30/60/90/180 day windows | Very High |
| **Derived indices** | Fire Weather Index composite | Current month | High |
| **Water year** | Oct-Sep precipitation total | Annual | Moderate |

### Key Findings

1. **VPD is the #1 predictor** of annual burned area (R² ≈ 0.25-0.40 depending on time window)
2. **Fire-month climate differs significantly** from background (KS tests p < 0.001 for VPD, Tmax, PPT)
3. **Antecedent climate matters**: Spring drought correlates with summer fire activity at 2-5 month lags
4. **Fire season VPD is trending upward**: +0.5-1.0 hPa/decade since 1950
5. **2020 was climatically extreme**: Record summer VPD + below-normal precipitation
6. **Burn severity scales with VPD**: Higher VPD risk classes show higher mean dNBR thresholds
7. **Spatial pattern**: Highest fire-climate risk in Southern CA mountains, Sierra foothills, and North Coast

### Integration with Previous Analyses

| Dataset | Key Climate Interaction |
|---------|----------------------|
| **CAL FIRE perimeters** | Annual burned area strongly tracks summer VPD |
| **MTBS severity** | dNBR thresholds modestly correlate with fire-month VPD |
| **FPA FOD** | Small fire frequency also scales with VPD (not shown — slower signal) |
| **Topography** | VPD × slope interaction: steep + dry = highest risk |
| **Vegetation** | Chaparral + high VPD = extreme fire risk zones |
| **Canopy Cover** | Dense forest + drought = crown fire potential |

### Next Steps

1. **Phase 2: Grid Creation** — Assign PRISM features to 800m × 800m grid cells
2. **Feature engineering** — Compute rolling windows, anomalies, and interaction terms at grid scale
3. **ML model** — CNN/LSTM with climate as primary temporal predictor

In [20]:
# ── Final Summary Statistics ──────────────────────────────────────────────

print("=" * 70)
print("PRISM Climate-Fire Integration — Part 3 Complete")
print("=" * 70)

import os
figure_files = sorted(FIGURE_DIR.glob("*.png"))
print(f"\nFigures generated: {len(figure_files)}")
for f in figure_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name} ({size_kb:.0f} KB)")

print(f"\nFire datasets analyzed:")
print(f"  CAL FIRE: {len(calfire_hq):,} fires (1993+)")
print(f"  MTBS: {len(mtbs):,} CA fires with {mtbs.has_severity.sum()} severity records")
print(f"  FPA FOD: {len(fpa):,} CA fires ({len(fpa_large):,} ≥100 acres)")
print(f"\nClimate extraction: {fire_climate['vpdmax'].notna().sum()} fires with VPD data")
print(f"Climate-severity overlay: {len(fire_sev)} fires with both climate and dNBR")

if ca_stats is not None:
    vpd_trend = ca_stats[(ca_stats['variable'] == 'vpdmax') &
                         (ca_stats['month'].isin([6,7,8]))].groupby('year')['mean'].mean()
    if len(vpd_trend) > 10:
        slope, _, r, p, _ = stats.linregress(vpd_trend.index.astype(float), vpd_trend.values)
        print(f"\nFire season VPD trend: {slope*10:+.2f} hPa/decade (p={p:.4f})")

print("\n✓ All 16 figures saved to outputs/figures/prism_fire/")
print("✓ Analysis complete — ready for Phase 2 grid integration")

PRISM Climate-Fire Integration — Part 3 Complete

Figures generated: 16
  01_temporal_alignment.png (425 KB)
  02_climate_at_fire_locations.png (556 KB)
  03_burned_vs_unburned_climate.png (528 KB)
  04_fire_location_climate_map.png (1908 KB)
  05_fire_year_climate.png (758 KB)
  06_annual_fire_climate_correlation.png (601 KB)
  07_worst_year_climate_anatomy.png (1014 KB)
  08_monthly_fire_climate_coupling.png (637 KB)
  09_fire_season_climate_trends.png (1146 KB)
  10_fire_season_vpd_vs_acres.png (295 KB)
  11_antecedent_climate_signals.png (782 KB)
  12_spring_drought_summer_fire.png (613 KB)
  13_cumulative_vpd_fire_risk.png (405 KB)
  14_severity_vs_climate.png (1428 KB)
  15_severity_climate_trends.png (473 KB)
  16_climate_fire_spatial_patterns.png (1665 KB)

Fire datasets analyzed:
  CAL FIRE: 9,902 fires (1993+)
  MTBS: 2,051 CA fires with 1581 severity records
  FPA FOD: 251,881 CA fires (5,428 ≥100 acres)

Climate extraction: 2051 fires with VPD data
Climate-severity overlay: